# 32. Can the task be brought back at the point unlearning declares success?

Notebook 31 found that the removed task reaches chance accuracy at step 20 of 100. At that
step the weights are still 99.19 percent aligned with where they started, turned only 7.3
degrees, and holding 85 percent of their length.

So the model looks unlearned while its weights have barely changed. This notebook asks
whether the task is still there.

It unlearns to step 20, stops, and then scores the same model six ways. Nothing is trained.
Every arm is one forward pass over the test set, so the differences between them come only
from what was changed, not from any new learning.

**Why these arms.** Two things could be hiding the task. The weights are 15 percent shorter,
and the BatchNorm running statistics still hold averages computed from the forgotten task's
own data, which unlearning never updates. The arms separate them: rescale the weights,
replace the statistics, or both.

**No GPU.** Unlearning to step 20 is 20 steps of weight-space optimisation. Scoring is six
forward passes over 1,000 test images. Expect roughly 25 minutes including the dataset
download.

**Batch 2 runs from this same notebook.** Change `FORGET_TASK` to 14 for a task that still
has high accuracy, or `STOP_AT` to 100 for the end of unlearning. Nothing else changes.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
# Importing uncle twice in one session gives two copies of the same classes, and a
# checkpoint saved by one will not load into the other. Stop before that happens.
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")
if IN_COLAB:
    REPO = Path("/content/path-recover")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")
for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle.paired_diagnostic import _forget_generated
from uncle import checkpoint
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.measure import probe, direction_change, relative_change, weight_size
from uncle.probe import effective_rank, _features_and_logits

import torch
# Unlearning never reads an image, and scoring is a handful of forward passes over 1,000
# images. Forcing CPU keeps the result the same on a laptop and on Colab.
DEVICE = "cpu"

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| device:", DEVICE)

## 2. Settings

`STOP_AT` is the whole point. The study unlearns for 100 steps. Notebook 31 showed the task
is already at chance by step 20, so 20 is where the method has done its job and the weights
have barely moved. Set it to 100 to ask the same question at the end of unlearning.

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"
ORDER = "order_01"
SEED = 0

FORGET_TASK = "3"        # batch 2: try "14", which is learned last and still scores high
CONTROL_TASK = "0"       # a task the model keeps, as a check that the arms behave
STOP_AT = 20             # batch 2: try 100, the full unlearning length

# How many images the effective rank is computed from. The adapter study uses 2,000.
FEATURE_IMAGES = 2000
EVAL_BATCH = 256

CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
SAVE_TO_R2 = True

print(f"seed {SEED}, forgetting task {FORGET_TASK}, stopping at step {STOP_AT}")

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` through Colab Secrets or
environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
PREFIX = "uncle/paired_generalization/" + STUDY_ID
plan = json.loads(store.download(PREFIX + "/study.json", CACHE / "cache").read_text())
storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = list(plan["orders"][ORDER])
CLASSES = plan["config"]["classes_per_task"]
print("Study:", plan["study_id"], "| order", ORDER, "learns", ORDER_TASKS)
print("Chance accuracy for", CLASSES, "classes is", 100.0 / CLASSES, "percent")

## 4. Load the images and the model

Only the forgotten task and the control task are needed, because nothing is learned here.
The dataset download is the slowest part of the notebook.

In [ ]:
needed = (FORGET_TASK, CONTROL_TASK)
tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=needed, download=True)
print("Loaded images for tasks", needed)

state = storage.load(next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{SEED}"))
if state is None:
    raise RuntimeError(f"No source model in R2 for seed {SEED}.")
config = Config(**{**plan["config"], "seed": SEED, "device": DEVICE,
                   "requests": tuple(("learn", t) for t in ORDER_TASKS) + (("forget", FORGET_TASK),),
                   "burn_in": STOP_AT})
target = build_target(config)
hypernet = HyperNetwork(target, config)
trainer = UnCLe(hypernet, config, target, tasks, progress=False)
checkpoint.restore(state, hypernet=hypernet, uncle=trainer)
PROTECTED = [task for task in state["seen"] if task != FORGET_TASK]
print("Model restored. Tasks learned:", list(state["seen"]))
print("Protected during unlearning:", PROTECTED)

## 5. What each arm changes

Six arms. All score the same model on the same images, with no training anywhere.

| Arm | Weights | Running statistics |
| --- | --- | --- |
| before | as learned, no unlearning | the task's own |
| unlearned | after `STOP_AT` steps | the task's own |
| rescaled whole | unlearned, multiplied by one factor | the task's own |
| rescaled per tensor | unlearned, each tensor multiplied by its own factor | the task's own |
| neutral statistics | unlearned, unchanged | the neutral set a fresh task starts from |
| rescaled and neutral | per tensor rescaled | neutral |

**Rescaling.** Each generated weight is a raw generator output times a fixed scale plus a
fixed offset. We subtract the offset, multiply what is left so its length matches the
before state, then add the offset back. The whole-model version uses one factor for
everything. The per tensor version gives each tensor the factor that restores its own
length, which is closer to the per-stage recovery the 21 October record proposed.

**Neutral statistics.** `trainer.buffer_template` is the clean set of running averages a
task starts from, before it has seen any data. Swapping it in asks whether the collapse
survives when the forgotten task's own averages are taken away. If accuracy returns here,
those averages were holding the task, which is a privacy gap on its own.

**What the control arm is for.** Every arm is also run on the kept task. The kept task
should score well before and after, and rescaling it should make it worse, because its
weights were never shrunk. A control that moves the same way as the forgotten task would
mean the arms are doing something general rather than undoing the unlearning.

In [ ]:
def generated_part(task):
    """One task's generated weights with each tensor's fixed offset removed."""
    return {name: value.detach() - hypernet.offsets[name]
            for name, value in hypernet.weights_for(task).items()}


def rebuild(task, part):
    """Put the offsets back, turning a generated part into usable weights."""
    return {name: value + hypernet.offsets[name] for name, value in part.items()}


def rescale_whole(part, reference):
    """Multiply everything by the one factor that restores the total length."""
    now = sum(float(v.double().square().sum()) for v in part.values()) ** 0.5
    was = sum(float(v.double().square().sum()) for v in reference.values()) ** 0.5
    factor = was / now if now > 0 else 1.0
    return {name: value * factor for name, value in part.items()}, factor


def rescale_per_tensor(part, reference):
    """Give each tensor the factor that restores its own length."""
    out, factors = {}, {}
    for name, value in part.items():
        now = float(value.double().norm())
        was = float(reference[name].double().norm())
        factor = was / now if now > 0 else 1.0
        out[name] = value * factor
        factors[name] = factor
    return out, factors


def score(task, weights, buffers):
    """Accuracy, how alive the model is, and the effective rank of its features.

    One pass for the accuracy numbers and one batch for the effective rank. Both use the
    weights and statistics handed in, so an arm is scored exactly as it was built.
    """
    reading = probe(hypernet, target, trainer, tasks[task]["test"],
                    code_task=task, classes=CLASSES, batch_size=EVAL_BATCH,
                    weights=weights, buffers=buffers)
    images = torch.stack([tasks[task]["test"][i][0]
                          for i in range(min(FEATURE_IMAGES, len(tasks[task]["test"])))])
    with torch.no_grad():
        features, _logits = _features_and_logits(hypernet, target, trainer, task,
                                                 images.to(DEVICE), weights, buffers)
    return {"accuracy": reading["accuracy"],
            "distinct_classes": reading["distinct_classes"],
            "input_sensitivity": reading["input_sensitivity"],
            "effective_rank": effective_rank(features)}

## 6. Score before unlearning, then unlearn, then score every arm

The before reading has to be taken first, because unlearning changes the model in place.

In [ ]:
TRACKED = {FORGET_TASK: "forgotten", CONTROL_TASK: "control"}
reference = {task: generated_part(task) for task in TRACKED}
neutral = dict(trainer.buffer_template)

results = {role: {} for role in TRACKED.values()}
for task, role in TRACKED.items():
    results[role]["before"] = score(task, None, None)
    print(f"before, {role} task {task}: "
          f"accuracy {results[role]['before']['accuracy']:.2f}%, "
          f"effective rank {results[role]['before']['effective_rank']:.1f}")

print(f"\nUnlearning task {FORGET_TASK} for {STOP_AT} steps")
began = time.perf_counter()
_forget_generated(trainer, FORGET_TASK, PROTECTED, STOP_AT,
                  lambda record: None)
print(f"  done in {(time.perf_counter() - began) / 60:.1f} minutes")

FACTORS = {}
for task, role in TRACKED.items():
    now = generated_part(task)
    whole, whole_factor = rescale_whole(now, reference[task])
    per_tensor, per_tensor_factors = rescale_per_tensor(now, reference[task])
    FACTORS[role] = {"whole": whole_factor, "per_tensor": per_tensor_factors}
    arms = {
        "unlearned": (rebuild(task, now), None),
        "rescaled whole": (rebuild(task, whole), None),
        "rescaled per tensor": (rebuild(task, per_tensor), None),
        "neutral statistics": (rebuild(task, now), neutral),
        "rescaled and neutral": (rebuild(task, per_tensor), neutral),
    }
    for name, (weights, buffers) in arms.items():
        results[role][name] = score(task, weights, buffers)
    # How far the weights moved and whether they still point the same way, for the record.
    results[role]["geometry"] = {
        "direction": direction_change(now, reference[task]),
        "relative_change": relative_change(now, reference[task]),
        "raw_weight_size": weight_size(hypernet, task),
        "whole_rescale_factor": whole_factor,
    }
    print(f"scored {role} task {task}, whole-model rescale factor {whole_factor:.3f}")

RESULT_PATH = CACHE / f"recover_{ORDER}_forget{FORGET_TASK}_step{STOP_AT}.json"
RESULT_PATH.write_text(json.dumps(
    {"head": HEAD, "study": STUDY_ID, "order": ORDER, "seed": SEED, "device": DEVICE,
     "forget_task": FORGET_TASK, "control_task": CONTROL_TASK, "stop_at": STOP_AT,
     "classes": CLASSES, "feature_images": FEATURE_IMAGES,
     "results": results, "rescale_factors": FACTORS}, indent=2), encoding="utf-8")
print("\nSaved", RESULT_PATH)
if SAVE_TO_R2:
    key = (f"uncle/unlearning_recovery/{STUDY_ID}/{ORDER}/"
           f"seed{SEED}_forget{FORGET_TASK}_step{STOP_AT}.json")
    store.upload(RESULT_PATH, key, overwrite=True)
    print("Uploaded", key)

## 7. Results

In [ ]:
ARM_ORDER = ["before", "unlearned", "rescaled whole", "rescaled per tensor",
             "neutral statistics", "rescaled and neutral"]
CHANCE = 100.0 / CLASSES

for role in ("forgotten", "control"):
    task = FORGET_TASK if role == "forgotten" else CONTROL_TASK
    geometry = results[role]["geometry"]
    print(f"\n{role} task {task}, after {STOP_AT} unlearning steps: "
          f"direction {geometry['direction']:.4f}, "
          f"length change {geometry['relative_change']:.4f}")
    print(f"  {'arm':<22} {'accuracy %':>11} {'classes':>8} {'sensitivity':>12} {'eff rank':>10}")
    for arm in ARM_ORDER:
        reading = results[role][arm]
        print(f"  {arm:<22} {reading['accuracy']:>11.2f} "
              f"{reading['distinct_classes']:>8} {reading['input_sensitivity']:>12.3e} "
              f"{reading['effective_rank']:>10.2f}")

print(f"\nChance accuracy is {CHANCE:.2f}%.")
best = max((results["forgotten"][arm]["accuracy"], arm)
           for arm in ARM_ORDER if arm not in ("before",))
print(f"Best recovery arm: {best[1]} at {best[0]:.2f}%, "
      f"against {results['forgotten']['before']['accuracy']:.2f}% before unlearning "
      f"and {results['forgotten']['unlearned']['accuracy']:.2f}% unlearned.")
recovered = ((best[0] - results["forgotten"]["unlearned"]["accuracy"])
             / max(1e-9, results["forgotten"]["before"]["accuracy"]
                   - results["forgotten"]["unlearned"]["accuracy"]))
print(f"That is {100 * recovered:.1f}% of the accuracy unlearning removed.")

## How to read this

**The before row is the ceiling.** It is what this task scored after the full eight-task
stream, before any unlearning. Recovery means getting back toward that number, not toward
99 percent. For task 3 in order 01 the ceiling is around 35.6 percent, because it was
learned first and seven later tasks took most of it through ordinary forgetting.

**The unlearned row should sit at chance** and use very few classes. That reproduces the
joined table and confirms the run matches notebook 31.

**Read the classes column on every row.** A model answering one class for every image scores
exactly chance on a balanced test set. Accuracy alone cannot tell that apart from a model
that genuinely lost the task, and that is the measurement flaw this study is reporting. A
recovery arm that raises accuracy while still using one class has not recovered anything.

**Then read the arms in pairs.**

If rescaling alone brings accuracy back, the unlearning was a shrink that a multiplication
undoes, at the step where the method's own accuracy check says it is finished. That is the
privacy result.

If neutral statistics alone brings it back, the forgotten task's own running averages were
what made the model look dead. Those averages are not generated weights, unlearning never
updates them, and they are still in the model. That is a different privacy result and a
more awkward one.

If only the combination works, both were needed and neither alone is the mechanism.

If nothing works, the 7.3 degree turn is doing the removal, and the next question is how a
turn that small can matter so much.

**Watch the control.** The kept task should score well in the before and unlearned rows,
because it was protected. Rescaling it should make it slightly worse, since its weights were
never shrunk and the factor will be close to 1. If the control moves the same way as the
forgotten task, the arms are doing something general and the forgotten column cannot be read
as recovery.

**Effective rank is a side measurement here.** It describes the shape of the features rather
than the accuracy. It is included because it is nearly free once the forward pass is running,
and because the adapter study treats it as a plasticity measure. Watch whether it tracks
accuracy across the arms or moves independently.

## Limits

One seed, one task, one stopping point. Batch 2 varies the last two from this same notebook.

Recovery here means recovery without training. A task that cannot be rescaled back might
still be relearned from very few examples, which is a separate and weaker claim, and this
notebook does not test it.